# POC — Mini Company Knowledge Assistant

**Goal:** build one small, runnable project that uses *every* concept from notebooks `01`–`07`.

This is a chatbot that can answer questions about a (fake) company handbook **and** do math — by combining **RAG** (retrieval) with **tool calling**.

### What each part teaches

| Part | Concept (notebook) | What you build |
|------|--------------------|----------------|
| 1 | LLM basics, tokens, temperature (`01`) + LLM API (`06`) | A function that talks to a real LLM |
| 2 | Structured outputs (`06`) | Force the LLM to return clean JSON |
| 3 | Embeddings & similarity (`02`) | Turn text into vectors, compare them |
| 4 | Chunking (`04`) | Split a document into searchable pieces |
| 5 | Vector database (`03`) | Store vectors + top-k similarity search |
| 6 | RAG (`04`) + Prompt engineering (`05`) | Retrieve -> augment -> generate |
| 7 | Tool calling (`07`) | Let the LLM run a calculator & a lookup tool |
| 8 | Everything together | The final assistant |

### The big picture

```
              +--------------- Your question ----------------+
              v                                              |
        Embed question  -->  Vector DB  -->  Top-K chunks    |
              |                                  |           |
              |                                  v           |
              |                         Augmented prompt     |
              v                                  |           |
            LLM  <--------------------------------           |
              |                                              |
       needs a tool? --yes--> run tool (calc / lookup) ------+
              | no
              v
         Final answer
```

> **How to use this notebook:** read each markdown cell, then run the code cell under it. Don't rush. Run, read the output, tweak a value, run again. That experimentation is where the learning happens.

## Part 0 — Setup

We need three things:

1. **`groq`** — the LLM client (free key, OpenAI-compatible API).
2. **`sentence-transformers`** — a *local* embedding model. First run downloads ~90 MB, then it works offline.
3. **A Groq API key** — get one free at [console.groq.com](https://console.groq.com/keys).

### Where to put your key (don't hard-code it!)

Recall from notebook `06`: **secrets go in environment variables, never in code.**

Create a file named `.env` in this `AI/` folder with one line:

```
GROQ_API_KEY=gsk_your_key_here
```

`.env` is already ignored by git, so your key stays private. The cell below loads it.

In [1]:
# Run this once to install the libraries (safe to re-run).
# The sentence-transformers install is large because it pulls in PyTorch.
#
# NOTE: versions are PINNED on purpose. The latest sentence-transformers/transformers
# require PyTorch >= 2.4. If you have an older torch (e.g. 2.3.x), the unpinned latest
# versions crash on import with "NameError: name 'nn' is not defined". These pins work
# with torch >= 2.0, so they're safe regardless of your torch version.
%pip install groq==0.11.0 "sentence-transformers==3.0.1" "transformers==4.44.2" python-dotenv numpy

  Attempting uninstall: groq
    Found existing installation: groq 1.5.0
    Uninstalling groq-1.5.0:
      Successfully uninstalled groq-1.5.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()  # reads AI/.env into environment variables

api_key = os.environ.get("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "No GROQ_API_KEY found. Create a file 'AI/.env' with:\n"
        "GROQ_API_KEY=gsk_your_key_here\n"
        "Get a free key at https://console.groq.com/keys"
    )

client = Groq(api_key=api_key)

# A current Groq chat model. If this name ever stops working, pick another from
# https://console.groq.com/docs/models and change it here.
#
# NOTE on tool calling (Part 7): the Llama models (e.g. "llama-3.3-70b-versatile")
# emit tool calls as TEXT markup that Groq then parses into JSON. They sometimes
# emit malformed markup, which makes Groq reject the request with a 400
# "tool_use_failed" error. The gpt-oss models do native tool calling and are far
# more reliable for the tool-calling loop, so we use one here.
MODEL = "openai/gpt-oss-20b"

print("Groq client ready. Using model:", MODEL)

Groq client ready. Using model: openai/gpt-oss-20b


## Part 1 — Talk to the LLM  *(notebooks 01 & 06)*

Remember the key ideas:

- An LLM API is **just an HTTP request/response** (notebook `06`).
- It is **stateless** — it does NOT remember past calls. *You* resend the messages each time.
- **Messages have roles**: `system` (instructions), `user` (the question), `assistant` (the reply).
- **`temperature`** controls randomness (notebook `01`): low = focused/factual, high = creative.
- **`usage`** tells you the token count = your bill (notebook `01`).

Let's write one helper, `ask_llm`, and watch all of this happen.

In [29]:
def ask_llm(question, system="You are a helpful, concise assistant.", temperature=0.2):
    """Send one question to the LLM and return its text answer."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": question},
        ],
        temperature=temperature,
    )
    answer = response.choices[0].message.content
    usage = response.usage
    print(f"[tokens] prompt={usage.prompt_tokens}  "
          f"completion={usage.completion_tokens}  total={usage.total_tokens}")
    return answer


print(ask_llm("In one sentence, what is Django?"))

[tokens] prompt=90  completion=86  total=176
Django is a high‑level Python web framework that encourages rapid development and clean, pragmatic design.


In [36]:
# Experiment: same prompt, different temperatures. Run a few times and compare.
print("--- temperature = 0.0 (focused / repeatable) ---")
print(ask_llm("Give me a name for a coffee shop.", temperature=0.0))

print("\n--- temperature = 1.2 (creative / varied) ---")
print(ask_llm("Give me a name for a coffee shop.", temperature=1.2))

# Try it yourself:
#  - Run this cell 3 times. Notice temp=0.0 stays similar, temp=1.2 changes a lot.
#  - This is exactly the temperature concept from notebook 01, but live.

--- temperature = 0.0 (focused / repeatable) ---
[tokens] prompt=91  completion=172  total=263
**Brewed Awakening**

--- temperature = 1.2 (creative / varied) ---
[tokens] prompt=91  completion=199  total=290
**Java Junction**


## Part 2 — Structured Outputs  *(notebook 06)*

Free-form text is great for humans but painful for code. In a backend you want **data**, not prose:

```
"Sure! Alex is 29 and reachable at alex@mail.com"   <- hard to parse
{ "name": "Alex", "age": 29, "email": "alex@mail.com" }   <- easy to use
```

Two tools from notebook `06`:

- **JSON mode** (`response_format={"type": "json_object"}`) — guarantees valid JSON *syntax*.
- **Pydantic validation** — guarantees the *shape* (right keys, right types).

We combine both: ask for JSON, then validate it with a Pydantic model. If it doesn't match, we know immediately instead of crashing later.

In [37]:
import json
from pydantic import BaseModel


class Person(BaseModel):
    name: str
    age: int
    email: str


def extract_person(text):
    """Use JSON mode + Pydantic to turn messy text into a typed Person object."""
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},  # JSON mode: valid JSON guaranteed
        messages=[
            {"role": "system", "content":
                "Extract the person's details. Respond ONLY with a JSON object "
                "having exactly these keys: name (string), age (integer), email (string)."},
            {"role": "user", "content": text},
        ],
    )
    raw = response.choices[0].message.content
    data = json.loads(raw)          # step 1: parse JSON (syntax)
    person = Person(**data)         # step 2: validate shape + types (Pydantic)
    return person


p = extract_person("Hi, I'm Alex, 29 years old, reach me at alex@mail.com")
print(p)
print("Just the age, already an int:", p.age + 1)

name='Alex' age=29 email='alex@mail.com'
Just the age, already an int: 30


## Part 3 — Embeddings & Similarity  *(notebook 02)*

An **embedding** turns text into a list of numbers (a vector) that captures its *meaning*. Texts with similar meaning end up as vectors that point in a similar direction.

We measure "similar direction" with **cosine similarity**:

```
cosine = 1.0   -> identical meaning
cosine ~ 0.0   -> unrelated
```

We use a small **local** model (`all-MiniLM-L6-v2`). The first run downloads it (~90 MB); after that it's offline and free. Each piece of text becomes a **384-dimension** vector.

In [38]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")  # downloads on first run


def embed(texts):
    """Turn a string (or list of strings) into vector(s)."""
    return embedder.encode(texts)


vec = embed("How many vacation days do I get?")
print("A single embedding is a vector of length:", len(vec))
print("First 5 numbers:", np.round(vec[:5], 3))

A single embedding is a vector of length: 384
First 5 numbers: [ 0.049  0.011  0.03   0.069 -0.034]


In [39]:
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


query = "How many holidays do employees get?"
candidates = [
    "Staff are entitled to 20 days of paid annual leave.",   # same meaning
    "The office cafeteria serves lunch from 12 to 2 pm.",     # unrelated
    "Workers receive twenty vacation days each year.",        # same meaning, different words
]

q_vec = embed(query)
for text in candidates:
    score = cosine_similarity(q_vec, embed(text))
    print(f"{score:.3f}   {text}")

# Notice: the two leave-related sentences score HIGH even though they share
# almost no words with the query. That's semantic search - matching meaning,
# not keywords. This is the engine behind retrieval.

0.510   Staff are entitled to 20 days of paid annual leave.
0.339   The office cafeteria serves lunch from 12 to 2 pm.
0.704   Workers receive twenty vacation days each year.


## Part 4 — Chunking  *(notebook 04)*

Our "knowledge source" is a small fake company handbook. We can't embed the whole thing as one vector — that would blur all topics together (notebook `04`). So we **chunk** it into smaller, focused pieces, with a little **overlap** so ideas don't get cut in half at a boundary.

```
chunk size     -> how big each piece is
chunk overlap  -> how much neighbours share (keeps boundary ideas whole)
```

For simplicity we chunk by paragraph here (our handbook is already written in clean paragraphs). In real projects you'd chunk by a fixed number of characters/tokens with overlap.

In [40]:
HANDBOOK = """
Annual Leave: Full-time employees receive 20 paid annual leave days per year.
Unused leave can be carried forward, up to a maximum of 5 days into the next year.

Work From Home: Employees may work from home up to 3 days per week.
Remote work must be agreed with your manager and logged in the HR portal.

Sick Leave: Employees get 10 paid sick days per year. A doctor's note is required
for any absence longer than 3 consecutive days.

Probation Period: New employees have a probation period of 6 months.
During probation, the notice period for resignation is 2 weeks.

Notice Period: After probation, the standard notice period for resignation is 30 days.

Reimbursements: Work-related expenses are reimbursed within 14 business days
after submitting receipts through the finance portal. The monthly limit is 15000.

Health Insurance: The company provides health insurance covering the employee,
a spouse, and up to two children. Coverage starts from the first working day.
"""


def chunk_text(text):
    """Split into paragraph chunks (separated by blank lines)."""
    paragraphs = [p.strip() for p in text.strip().split("\n\n")]
    return [p for p in paragraphs if p]


chunks = chunk_text(HANDBOOK)
print(f"Handbook split into {len(chunks)} chunks:\n")
for i, c in enumerate(chunks):
    print(f"[{i}] {c[:60]}...")

Handbook split into 7 chunks:

[0] Annual Leave: Full-time employees receive 20 paid annual lea...
[1] Work From Home: Employees may work from home up to 3 days pe...
[2] Sick Leave: Employees get 10 paid sick days per year. A doct...
[3] Probation Period: New employees have a probation period of 6...
[4] Notice Period: After probation, the standard notice period f...
[5] Reimbursements: Work-related expenses are reimbursed within ...
[6] Health Insurance: The company provides health insurance cove...


## Part 5 — A Tiny Vector Database  *(notebook 03)*

A vector database does three jobs (notebook `03`):

1. **Store** vectors (+ the original text).
2. **Search** for the vectors most similar to a query.
3. **Return** the top matches (**top-k**).

Real products (Chroma, Pinecone, Qdrant) add indexing, persistence, and scale. But the *core idea* is just "store vectors and find the nearest ones." We'll build a minimal in-memory version so you can see exactly what's happening — no magic.

In [41]:
class SimpleVectorStore:
    """A minimal in-memory vector DB: stores texts + their embeddings, searches by cosine similarity."""

    def __init__(self):
        self.texts = []
        self.vectors = []

    def add(self, texts):
        vectors = embed(texts)
        for text, vec in zip(texts, vectors):
            self.texts.append(text)
            self.vectors.append(vec)

    def search(self, query, top_k=3):
        q_vec = embed(query)
        scored = [
            (cosine_similarity(q_vec, vec), text)
            for text, vec in zip(self.texts, self.vectors)
        ]
        scored.sort(reverse=True)          # highest similarity first
        return scored[:top_k]              # the top-k matches


store = SimpleVectorStore()
store.add(chunks)
print(f"Stored {len(store.texts)} chunks in the vector DB.\n")

print('Top-3 matches for: "How much vacation do I get?"\n')
for score, text in store.search("How much vacation do I get?", top_k=3):
    print(f"{score:.3f}  ->  {text[:70]}...")

Stored 7 chunks in the vector DB.

Top-3 matches for: "How much vacation do I get?"

0.483  ->  Annual Leave: Full-time employees receive 20 paid annual leave days pe...
0.426  ->  Sick Leave: Employees get 10 paid sick days per year. A doctor's note ...
0.285  ->  Reimbursements: Work-related expenses are reimbursed within 14 busines...


## Part 6 — RAG: Retrieve, Augment, Generate  *(notebooks 04 & 05)*

Now we connect the pieces. RAG = three steps:

```
1. Retrieve   ->  find the top-k relevant chunks (Part 5)
2. Augment    ->  put those chunks INTO the prompt (prompt engineering, notebook 05)
3. Generate   ->  ask the LLM to answer USING ONLY that context
```

The **system prompt** is where prompt engineering matters: we tell the model to answer *only* from the provided context and to say "I don't know" otherwise. This is the main defense against **hallucination** (notebook `01`).

Compare the two answers below: the LLM **without** context vs **with** retrieved context.

In [42]:
RAG_SYSTEM = (
    "You are a company HR assistant. Answer the user's question using ONLY the "
    "context provided. If the answer is not in the context, say 'I don't know based "
    "on the handbook.' Be concise."
)


def rag_answer(question, top_k=3, show_context=False):
    # 1. RETRIEVE
    hits = store.search(question, top_k=top_k)
    context = "\n\n".join(text for _, text in hits)

    # 2. AUGMENT  (build a prompt that contains the retrieved context)
    augmented_user_prompt = f"Context:\n{context}\n\nQuestion: {question}"

    if show_context:
        print("--- Retrieved context ---")
        print(context)
        print("-------------------------\n")

    # 3. GENERATE
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": RAG_SYSTEM},
            {"role": "user", "content": augmented_user_prompt},
        ],
    )
    return response.choices[0].message.content


question = "Can I carry forward unused leave, and how much?"

print("WITHOUT context (plain LLM, may hallucinate):")
print(ask_llm(question))

print("\nWITH RAG (grounded in the handbook):")
print(rag_answer(question, show_context=True))

WITHOUT context (plain LLM, may hallucinate):
[tokens] prompt=93  completion=210  total=303
I’m not sure which company’s policy you’re referring to. Could you let me know the organization (or the specific leave type) so I can give you the exact carry‑forward limit?

WITH RAG (grounded in the handbook):
--- Retrieved context ---
Annual Leave: Full-time employees receive 20 paid annual leave days per year.
Unused leave can be carried forward, up to a maximum of 5 days into the next year.

Sick Leave: Employees get 10 paid sick days per year. A doctor's note is required
for any absence longer than 3 consecutive days.

Reimbursements: Work-related expenses are reimbursed within 14 business days
after submitting receipts through the finance portal. The monthly limit is 15000.
-------------------------

Yes. You can carry forward unused annual leave, but only up to a maximum of **5 days** into the next year.


In [43]:
# A question the handbook does NOT cover. Good RAG should admit it doesn't know
# instead of making something up.
print(rag_answer("What is the company's policy on pet insurance?"))

I don't know based on the handbook.


## Part 7 — Tool Calling  *(notebook 07)*

RAG handles "what does the handbook say?" But some questions need **action** or **exact computation** the LLM can't do reliably — e.g. *"What's 3 months of a 15000 reimbursement limit?"*

A **tool** is a normal Python function + a **description the model can read**. The crucial rule from notebook `07`:

```
The LLM does NOT run tools.
YOUR code runs the tool and feeds the result back to the LLM.
```

The loop:

```
1. Send question + list of available tools.
2. Model replies with EITHER a final answer OR a tool call.
3. If tool call -> YOUR code runs it -> send the result back.
4. Repeat until the model gives a final answer.
```

We'll give the model two tools: a **calculator** and a **handbook search** (reusing our vector store).

In [44]:
import ast
import operator

# --- The actual Python functions (the tools) ---

# A SAFE calculator. We do NOT use raw eval() (notebook 07 warned about this).
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg,
}


def _safe_eval(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp):
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Unsupported expression")


def calculator(expression):
    """Evaluate a basic math expression like '15000 * 3'."""
    return _safe_eval(ast.parse(expression, mode="eval").body)


def search_handbook(query):
    """Return the most relevant handbook text for a query."""
    hits = store.search(query, top_k=2)
    return "\n".join(text for _, text in hits)


# Map tool names -> functions so we can dispatch by name later.
AVAILABLE_TOOLS = {
    "calculator": calculator,
    "search_handbook": search_handbook,
}

print("calculator('15000 * 3') =", calculator("15000 * 3"))
print("search_handbook('sick leave') ->", search_handbook("sick leave")[:60], "...")

calculator('15000 * 3') = 45000
search_handbook('sick leave') -> Sick Leave: Employees get 10 paid sick days per year. A doct ...


In [45]:
# --- The tool DESCRIPTIONS the model reads (this is the "interface" from notebook 07) ---
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a basic arithmetic expression and return the number. "
                           "Use this for any math instead of computing it yourself.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A math expression, e.g. '15000 * 3'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_handbook",
            "description": "Look up relevant information from the company employee handbook "
                           "(leave, work from home, sick days, notice period, reimbursements, insurance).",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "What to look up, e.g. 'reimbursement monthly limit'",
                    }
                },
                "required": ["query"],
            },
        },
    },
]

In [46]:
from groq import BadRequestError


def _create_with_retry(messages, max_retries=3):
    """Call the model, retrying if it emits a malformed tool call.

    Some models occasionally produce tool-call markup that Groq can't parse and
    reject with a 400 'tool_use_failed'. A small temperature bump on retry nudges
    the model toward a clean tool call instead of failing the whole demo.
    """
    for attempt in range(max_retries):
        try:
            return client.chat.completions.create(
                model=MODEL,
                temperature=0 if attempt == 0 else 0.4,
                messages=messages,
                tools=TOOL_SCHEMAS,    # tell the model what tools exist
                tool_choice="auto",    # let the model decide whether to use one
            )
        except BadRequestError as e:
            if "tool_use_failed" in str(e) and attempt < max_retries - 1:
                print(f"[retry {attempt + 1}] model produced a malformed tool call, retrying...")
                continue
            raise
    raise RuntimeError("Model kept producing malformed tool calls.")


def run_with_tools(question, system="You are a helpful company assistant.", max_steps=10, verbose=True):
    """The tool-calling loop: ask -> (maybe run tool) -> feed result back -> repeat -> answer."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": question},
    ]

    for step in range(max_steps):
        response = _create_with_retry(messages)
        msg = response.choices[0].message
        print(msg)

        # No tool call -> the model is done, return its answer.
        if not msg.tool_calls:
            return msg.content

        # The model asked to call one or more tools. We must run them.
        messages.append(msg)           # keep the model's tool-call request in history
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            if verbose:
                print(f"[step {step}] model called: {name}({args})")

            result = AVAILABLE_TOOLS[name](**args)   # OUR code runs the tool

            # Feed the result back as a 'tool' message.
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result),
            })

    return "Stopped: too many tool steps."


# A question that needs BOTH tools: look up the limit, then do math.
print(run_with_tools(
    "What is the total reimbursement limit over 3 months? Use the handbook for the monthly limit."
))

ChatCompletionMessage(content=None, role='assistant', function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='fc_7d8c6151-73fc-4d5b-8f8a-4776fd16ee2e', function=Function(arguments='{"query":"reimbursement monthly limit"}', name='search_handbook'), type='function')], reasoning='We need to look up monthly limit from handbook. Then compute total over 3 months. Use calculator tool. Let\'s search handbook for "reimbursement monthly limit".')
[step 0] model called: search_handbook({'query': 'reimbursement monthly limit'})
ChatCompletionMessage(content=None, role='assistant', function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='fc_40039982-a8c3-4561-9d3b-216591870f4e', function=Function(arguments='{"expression":"15000 * 3"}', name='calculator'), type='function')], reasoning='Monthly limit is 15000. Over 3 months: 15000 * 3 = 45000. Use calculator.')
[step 1] model called: calculator({'expression': '15000 * 3'})
ChatCompletionMessage(content='The monthly reimbursement li

## Part 8 — The Full Assistant

Notice something: in Part 7, `search_handbook` is itself a tool. That means our tool-calling loop **already includes RAG** — the model retrieves from the handbook *when it decides it needs to*, and uses the calculator *when math is needed*. 

This is the punchline of notebook `07`:

> **An agent is basically an LLM that can use tools.**

The cell below is our finished assistant — one function, backed by everything we built. Try the different example questions and watch which tools the model chooses.

In [47]:
ASSISTANT_SYSTEM = (
    "You are a friendly company HR assistant. "
    "Use search_handbook to answer policy questions, and calculator for any math. "
    "If the handbook doesn't cover something, say so honestly. Keep answers concise."
)


def assistant(question):
    print(f"USER: {question}\n")
    answer = run_with_tools(question, system=ASSISTANT_SYSTEM)
    print(f"\nASSISTANT: {answer}\n")
    print("=" * 70)


# Try each of these (and then write your own!):
assistant("How many sick days do I get and what if I'm off for a week?")
assistant("If I work from home the maximum allowed days, how many WFH days is that over 4 weeks?")
assistant("What's the notice period after probation?")
assistant("Do we get free gym membership?")   # not in the handbook -> should say it doesn't know

USER: How many sick days do I get and what if I'm off for a week?

ChatCompletionMessage(content=None, role='assistant', function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='fc_9b575e23-1155-4347-9dcb-dd20bca7cbb0', function=Function(arguments='{"query":"sick days policy"}', name='search_handbook'), type='function')], reasoning='We need to use search_handbook to find sick days policy. Then answer concisely.')
[step 0] model called: search_handbook({'query': 'sick days policy'})
ChatCompletionMessage(content='You’re entitled to **10 paid sick days per year**.  \nIf you’re off for a week (5 days), you’ll need to provide a **doctor’s note** because the absence exceeds 3 consecutive days.', role='assistant', function_call=None, tool_calls=None)

ASSISTANT: You’re entitled to **10 paid sick days per year**.  
If you’re off for a week (5 days), you’ll need to provide a **doctor’s note** because the absence exceeds 3 consecutive days.

USER: If I work from home the maximum allowe

## Recap — what you just built

```
Question
   |
   v
LLM with tools  --(needs handbook?)--> search_handbook --> embed query
   |                                         |              --> top-k vector search (RAG)
   |                                         v
   |<------------------------------- retrieved chunks
   |
   --(needs math?)--> calculator --> exact result
   |
   v
Grounded, accurate answer
```

Every concept from `01`–`07` is in here:

- **01** — you called a real LLM, set temperature, watched token usage, and saw how RAG + tools fight hallucination.
- **02** — you turned text into embeddings and compared them with cosine similarity.
- **03** — you built a vector store with top-k search.
- **04** — you chunked a document and ran the full retrieve -> augment -> generate pipeline.
- **05** — your system prompts shaped behaviour (answer only from context, admit when unsure).
- **06** — you called the LLM API and forced structured JSON output validated by Pydantic.
- **07** — you gave the LLM tools and ran the call -> execute -> feed-back loop.

### Exercises (this is where it really sticks)

1. **Add a tool.** Write `get_today_date()` and register it, then ask "How many days until my probation ends if I joined on the 1st?"
2. **Break RAG on purpose.** In Part 6, set `top_k=1` and ask a question whose answer spans two chunks. See retrieval miss context — then raise `top_k` and watch it recover.
3. **Tune the prompt.** Remove "say I don't know" from `RAG_SYSTEM` and re-ask the pet-insurance question. Watch it start to hallucinate.
4. **Real chunking.** Replace `chunk_text` with fixed-size character chunks (e.g. 200 chars, 40 overlap) and compare retrieval quality.
5. **Structured RAG.** Make `rag_answer` return a Pydantic object `{answer: str, source_chunk: str, confident: bool}` by combining Part 2 + Part 6.
6. **Swap the model.** Change `MODEL` to a smaller Groq model and compare speed, cost (tokens), and answer quality.

> Once these feel natural, you're ready for notebook `08` (Evaluation) — *how do you prove this assistant is actually correct?* — and then `09` Agents, which is just this tool loop with memory and planning.